In [20]:
!pip install -q groq langchain-groq
!pip install -q langchain langchain-core
!pip install -q langchain-community
!pip install -q faiss-cpu pypdf
!pip install -q sentence-transformers
!pip install -q langgraph
print("✅ All installed!")

✅ All installed!


In [21]:
!pip install -q langchain-huggingface
print("✅ Done!")

✅ Done!


In [22]:
import os
import warnings
warnings.filterwarnings('ignore')

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

GROQ_API_KEY = "your-groq-api-key-here"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

# Embeddings using sentence-transformers
embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ All libraries imported!")
print("✅ Groq LLM ready!")
print("✅ Embeddings model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ All libraries imported!
✅ Groq LLM ready!
✅ Embeddings model loaded!


In [23]:
# Cell 3 — Load and Chunk PDFs

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Path where your PDFs are uploaded in Colab ──
PDF_FOLDER = "/content"

# ── Find all PDF files ──
pdf_files = [f for f in os.listdir(PDF_FOLDER)
             if f.endswith('.pdf')]

print(f"📄 Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"   → {pdf}")

# ── Load all PDFs ──
all_documents = []
for pdf_file in pdf_files:
    pdf_path = os.path.join(PDF_FOLDER, pdf_file)
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    all_documents.extend(documents)
    print(f"✅ Loaded: {pdf_file} ({len(documents)} pages)")

print(f"\n📚 Total pages loaded: {len(all_documents)}")

# ── Split into chunks ──
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # 500 characters per chunk
    chunk_overlap=50,      # 50 chars overlap between chunks
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(all_documents)

print(f"✂️  Total chunks created: {len(chunks)}")
print(f"\n🔍 Sample chunk preview:")
print("-" * 50)
print(chunks[0].page_content[:300])
print("-" * 50)
print("✅ Step 3 Complete — PDFs loaded and chunked!")

📄 Found 3 PDF files:
   → Brochure_Arogya_Sanjeevani_Insurance_Policy_V_9_healthcare.pdf
   → Policy_Star_Comprehensive_Insurance_Policy_ Healthcare.pdf
   → Brochure_Family_Health_Optima_Insurance_Plan_healthcare.pdf
✅ Loaded: Brochure_Arogya_Sanjeevani_Insurance_Policy_V_9_healthcare.pdf (6 pages)
✅ Loaded: Policy_Star_Comprehensive_Insurance_Policy_ Healthcare.pdf (48 pages)
✅ Loaded: Brochure_Family_Health_Optima_Insurance_Plan_healthcare.pdf (14 pages)

📚 Total pages loaded: 68
✂️  Total chunks created: 378

🔍 Sample chunk preview:
--------------------------------------------------
SHAHLIP26045V042526
Star Health and Allied Insurance Co Ltd. Star Health and Allied Insurance Co Ltd. Star Health and Allied Insurance Co Ltd.
--------------------------------------------------
✅ Step 3 Complete — PDFs loaded and chunked!


In [24]:
# Cell 4 — Create FAISS Vector Database

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings

print("⏳ Creating embeddings for 378 chunks...")
print("   (This converts text → numbers for smart search)")
print("   Please wait 2-3 minutes...\n")

# Step 4a: Initialize embedding model
embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
# all-MiniLM-L6-v2:
# → Small but powerful embedding model
# → Free to use
# → Converts sentences into 384-dimensional vectors
# → Fast and accurate for search

# Step 4b: Create FAISS vector store from chunks
vectorstore = FAISS.from_documents(
    documents=chunks,      # our 378 chunks
    embedding=embeddings   # our embedding model
)

print("✅ Vector database created!")
print(f"✅ {len(chunks)} chunks converted to vectors")

# Step 4c: Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    # similarity = find chunks with similar MEANING
    search_kwargs={"k": 4}
    # k=4 = return top 4 most relevant chunks
)

print("✅ Retriever ready!")
print("\n🔍 Testing retriever with sample query...")

# Step 4d: Test it works
test_query = "What is covered under this insurance policy?"
test_results = retriever.invoke(test_query)

print(f"✅ Found {len(test_results)} relevant chunks for query:")
print(f"   '{test_query}'")
print("\n📄 Most relevant chunk found:")
print("-" * 50)
print(test_results[0].page_content[:300])
print("-" * 50)
print("\n✅ Step 4 Complete — FAISS Vector DB Ready!")

⏳ Creating embeddings for 378 chunks...
   (This converts text → numbers for smart search)
   Please wait 2-3 minutes...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector database created!
✅ 378 chunks converted to vectors
✅ Retriever ready!

🔍 Testing retriever with sample query...
✅ Found 4 relevant chunks for query:
   'What is covered under this insurance policy?'

📄 Most relevant chunk found:
--------------------------------------------------
Insured under this policy following after 
every claim free year up to a maximum 
of 100%.
--------------------------------------------------

✅ Step 4 Complete — FAISS Vector DB Ready!


In [25]:
# Cell 5 — Build RAG Pipeline

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Step 5a: Create the prompt template
# This tells the LLM HOW to answer
prompt_template = """You are a helpful healthcare
insurance assistant. Answer the question based ONLY
on the context provided from insurance policy documents.

If the answer is not in the context, say:
"I cannot find this information in the policy documents."

Context from policy documents:
{context}

Question: {question}

Provide a clear, accurate answer based on the policy:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Step 5b: Helper function to format retrieved chunks
def format_docs(docs):
    # Joins all retrieved chunks into one text block
    # So LLM can read all 4 chunks together
    return "\n\n".join(doc.page_content for doc in docs)

# Step 5c: Build the RAG chain
# This connects: retriever → prompt → llm → output
rag_chain = (
    {
        "context": retriever | format_docs,
        # retriever finds chunks, format_docs joins them
        "question": RunnablePassthrough()
        # passes user question directly through
    }
    | prompt      # fills the prompt template
    | llm         # sends to Groq LLM
    | StrOutputParser()  # converts response to string
)

print("✅ RAG Pipeline built successfully!")
print("✅ Prompt template ready!")
print("✅ RAG chain connected!")

# Step 5d: TEST the RAG pipeline
print("\n" + "="*50)
print("🧪 TESTING RAG PIPELINE")
print("="*50)

test_questions = [
    "What is the sum insured amount?",
    "Is hospitalization covered?",
    "What is the waiting period?"
]

for question in test_questions:
    print(f"\n❓ Question: {question}")
    print("⏳ Searching policy documents...")
    answer = rag_chain.invoke(question)
    print(f"💬 Answer: {answer}")
    print("-" * 40)

print("\n✅ Step 5 Complete — RAG Pipeline Working!")

✅ RAG Pipeline built successfully!
✅ Prompt template ready!
✅ RAG chain connected!

🧪 TESTING RAG PIPELINE

❓ Question: What is the sum insured amount?
⏳ Searching policy documents...
💬 Answer: The sum insured amount is not explicitly stated as a single value, but rather provided as a range of options. The available sum insured amounts are:

- Rs. 2,50,000/-
- Rs. 5,00,000/-
- Rs. 7,50,000/-
- Rs. 10,00,000/-
- Rs. 15,00,000/-
- Rs. 20,00,000/-
- Rs. 25,00,000/-
- Rs. 50,00,000/-
- Rs. 75,00,000/-
- Rs. 1,00,00,000/- 

Note that one specific sum insured amount mentioned is Rs. 5,00,000, which is related to the cumulative bonus calculation. However, this does not necessarily imply it is the only sum insured amount available.
----------------------------------------

❓ Question: Is hospitalization covered?
⏳ Searching policy documents...
💬 Answer: Yes, hospitalization is covered under this policy, as it is mentioned that "Hospitalization due to continuous period of illness including its 

In [26]:
# Cell 6 — Coverage Verification Agent

from typing import TypedDict
from langgraph.graph import StateGraph, END

# Step 6a: Define the Agent State
# State = memory that passes between agents
class ClaimsState(TypedDict):
    question: str        # user's original question
    question_type: str   # what type of question
    coverage_answer: str # answer from coverage agent
    claims_answer: str   # answer from claims agent
    final_answer: str    # final combined answer

# Step 6b: Coverage Verification Agent
def coverage_verification_agent(state: ClaimsState):
    """
    This agent specializes in:
    - What is covered/not covered
    - Coverage amounts and limits
    - Waiting periods
    - Policy inclusions/exclusions
    """

    question = state["question"]

    # Specialized prompt for coverage questions
    coverage_prompt = f"""You are a Coverage Verification
Specialist for healthcare insurance policies.

Your job is to verify if a treatment/procedure is covered.

Always structure your answer as:
COVERAGE STATUS: [YES/NO/PARTIAL]
DETAILS: [specific coverage details from policy]
CONDITIONS: [any conditions or waiting periods]
POLICY REFERENCE: [which document/section]

Based on the insurance policy documents, answer:
{question}"""

    # Use RAG to find relevant chunks first
    relevant_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in relevant_docs
    )

    # Then use LLM with context
    from langchain_core.messages import HumanMessage

    full_prompt = f"""Use this policy context:
{context}

Now answer as Coverage Verification Specialist:
{coverage_prompt}"""

    response = llm.invoke([HumanMessage(
        content=full_prompt
    )])

    # Update state with coverage answer
    return {
        **state,
        "coverage_answer": response.content
    }

# Step 6c: Test Coverage Agent alone
print("🧪 Testing Coverage Verification Agent...")
print("=" * 50)

test_state = {
    "question": "Is cataract surgery covered under the policy?",
    "question_type": "coverage",
    "coverage_answer": "",
    "claims_answer": "",
    "final_answer": ""
}

result = coverage_verification_agent(test_state)

print("❓ Question:", test_state["question"])
print("\n💬 Coverage Agent Answer:")
print("-" * 40)
print(result["coverage_answer"])
print("-" * 40)
print("\n✅ Step 6 Complete — Coverage Agent Working!")

🧪 Testing Coverage Verification Agent...
❓ Question: Is cataract surgery covered under the policy?

💬 Coverage Agent Answer:
----------------------------------------
COVERAGE STATUS: YES
DETAILS: Cataract treatment is covered up to the limits specified in the policy, with all day care procedures covered up to the Sum Insured, and in-patient procedures covered up to 50% of the Sum Insured. The limits per eye and per policy period are as follows: for a Sum Insured of ₹5,00,000, the limit is up to ₹40,000 per eye and up to ₹75,000 per policy period.
CONDITIONS: The policy is subject to a co-payment of 20% of each and every claim amount for insured persons whose age at the time of entry is 61 years and above.
POLICY REFERENCE: Cataract Treatment section of the policy document.
----------------------------------------

✅ Step 6 Complete — Coverage Agent Working!


In [27]:
# Cell 7 — Claims Status Agent

def claims_status_agent(state: ClaimsState):
    """
    This agent specializes in:
    - Claim filing procedures
    - Claim processing timelines
    - Required documentation
    - Claim rejection reasons
    - Reimbursement process
    """

    question = state["question"]

    # Specialized prompt for claims questions
    claims_prompt = f"""You are a Claims Processing
Specialist for healthcare insurance.

Your job is to help with claim status and procedures.

Always structure your answer as:
CLAIM TYPE: [Cashless/Reimbursement/Network]
PROCESS: [step by step claim process]
DOCUMENTS NEEDED: [list of required documents]
TIMELINE: [processing time]
IMPORTANT NOTE: [any critical information]

Based on the insurance policy documents, answer:
{question}"""

    # Use RAG to find relevant chunks
    relevant_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in relevant_docs
    )

    from langchain_core.messages import HumanMessage

    full_prompt = f"""Use this policy context:
{context}

Now answer as Claims Processing Specialist:
{claims_prompt}"""

    response = llm.invoke([HumanMessage(
        content=full_prompt
    )])

    return {
        **state,
        "claims_answer": response.content
    }

# Test Claims Agent
print("🧪 Testing Claims Status Agent...")
print("=" * 50)

test_state_2 = {
    "question": "What documents do I need to submit for a claim?",
    "question_type": "claims",
    "coverage_answer": "",
    "claims_answer": "",
    "final_answer": ""
}

result2 = claims_status_agent(test_state_2)

print("❓ Question:", test_state_2["question"])
print("\n💬 Claims Agent Answer:")
print("-" * 40)
print(result2["claims_answer"])
print("-" * 40)
print("\n✅ Step 7 Complete — Claims Agent Working!")

🧪 Testing Claims Status Agent...
❓ Question: What documents do I need to submit for a claim?

💬 Claims Agent Answer:
----------------------------------------
CLAIM TYPE: Reimbursement
PROCESS: The claim process involves notifying the company within the stipulated time limit, submitting the required documents, and awaiting the company's verification and settlement of the claim. The company may also require additional documents or an examination by an authorized doctor if necessary.
DOCUMENTS NEEDED: 
- For Accidental Death Claims: 
  1. Death Certificate
  2. Post-mortem Certificate (if conducted)
  3. FIR (wherever required)
  4. Police Investigation report (wherever required)
  5. Viscera Sample Report (wherever required)
  6. Forensic Science Laboratory report (wherever required)
  7. Legal Heir Certificate (wherever required)
  8. Succession Certificate (wherever required)
- For Permanent Total Disablement Claims: 
  1. Certificate from Government doctor confirming the disability an

In [28]:
# Cell 8 — Supervisor + LangGraph Workflow

from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

# Step 8a: Supervisor Agent
# Decides which specialist agent to call
def supervisor_agent(state: ClaimsState):
    """
    Supervisor reads the question and decides:
    → Send to Coverage Agent, OR
    → Send to Claims Agent
    """
    question = state["question"]

    # Ask LLM to classify the question
    classification_prompt = f"""You are a supervisor
for a healthcare insurance assistant.

Classify this question into ONE category only:
- "coverage" → questions about what is covered,
   benefits, waiting periods, inclusions, exclusions
- "claims" → questions about claim process,
   documents needed, how to file, claim status

Question: "{question}"

Reply with ONLY one word: coverage OR claims"""

    response = llm.invoke([HumanMessage(
        content=classification_prompt
    )])

    # Get the classification
    question_type = response.content.strip().lower()

    # Clean up response
    if "coverage" in question_type:
        question_type = "coverage"
    else:
        question_type = "claims"

    print(f"🎯 Supervisor Decision: → {question_type} agent")

    return {**state, "question_type": question_type}

# Step 8b: Router function
# Tells LangGraph which node to go to next
def route_question(state: ClaimsState):
    """Routes to correct agent based on question type"""
    return state["question_type"]

# Step 8c: Final answer compiler
def compile_final_answer(state: ClaimsState):
    """Compiles the final answer to show user"""

    if state["question_type"] == "coverage":
        final = state["coverage_answer"]
    else:
        final = state["claims_answer"]

    return {**state, "final_answer": final}

# Step 8d: Build the LangGraph workflow
print("🔨 Building LangGraph workflow...")

# Create the graph
workflow = StateGraph(ClaimsState)

# Add all nodes (agents) to the graph
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("coverage_agent",
                  coverage_verification_agent)
workflow.add_node("claims_agent",
                  claims_status_agent)
workflow.add_node("compile_answer",
                  compile_final_answer)

# Set entry point — always start with supervisor
workflow.set_entry_point("supervisor")

# Add conditional edges
# After supervisor → go to coverage OR claims
workflow.add_conditional_edges(
    "supervisor",        # from supervisor
    route_question,      # use this function to decide
    {
        "coverage": "coverage_agent",  # if coverage
        "claims": "claims_agent"       # if claims
    }
)

# After each agent → compile the answer
workflow.add_edge("coverage_agent", "compile_answer")
workflow.add_edge("claims_agent", "compile_answer")

# After compiling → END
workflow.add_edge("compile_answer", END)

# Compile the graph into runnable app
app = workflow.compile()

print("✅ LangGraph workflow built!")
print("\n📊 Workflow Structure:")
print("   START")
print("     ↓")
print("   Supervisor (classifies question)")
print("     ↓              ↓")
print("   Coverage      Claims")
print("   Agent         Agent")
print("     ↓              ↓")
print("   Compile Final Answer")
print("     ↓")
print("   END")
print("\n✅ Step 8 Complete — Multi-Agent System Ready!")

🔨 Building LangGraph workflow...
✅ LangGraph workflow built!

📊 Workflow Structure:
   START
     ↓
   Supervisor (classifies question)
     ↓              ↓
   Coverage      Claims
   Agent         Agent
     ↓              ↓
   Compile Final Answer
     ↓
   END

✅ Step 8 Complete — Multi-Agent System Ready!


In [29]:
# Cell 9 — FINAL SYSTEM TEST

print("=" * 60)
print("🏥 HEALTHCARE CLAIMS ASSISTANCE PLATFORM")
print("   Multi-Agent AI System — FINAL TEST")
print("=" * 60)

# Test questions covering both agents
test_questions = [
    # Coverage questions → should go to Coverage Agent
    "Is maternity benefit covered under the policy?",
    "What is the waiting period for pre-existing diseases?",

    # Claims questions → should go to Claims Agent
    "How do I file a reimbursement claim?",
    "What documents are needed for hospitalization claim?",

    # Tricky question → supervisor decides
    "Is there a co-payment for senior citizens?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}: {question}")
    print('='*60)

    # Create initial state
    initial_state = {
        "question": question,
        "question_type": "",
        "coverage_answer": "",
        "claims_answer": "",
        "final_answer": ""
    }

    # Run through complete multi-agent system
    result = app.invoke(initial_state)

    print(f"\n📋 FINAL ANSWER:")
    print("-" * 40)
    print(result["final_answer"])
    print("-" * 40)

print("\n" + "="*60)
print("✅ ALL TESTS COMPLETE!")
print("✅ Multi-Agent System Working Perfectly!")
print("="*60)
print("\n🎯 PROJECT SUMMARY:")
print("   ✅ 3 Insurance PDFs processed")
print("   ✅ 378 chunks in FAISS vector DB")
print("   ✅ RAG Pipeline with Groq LLM")
print("   ✅ Coverage Verification Agent")
print("   ✅ Claims Status Agent")
print("   ✅ LangGraph Supervisor workflow")
print("   ✅ Intelligent question routing")
print("\n🚀 Ready for GitHub upload!")

🏥 HEALTHCARE CLAIMS ASSISTANCE PLATFORM
   Multi-Agent AI System — FINAL TEST

TEST 1: Is maternity benefit covered under the policy?
🎯 Supervisor Decision: → coverage agent

📋 FINAL ANSWER:
----------------------------------------
COVERAGE STATUS: YES
DETAILS: The policy covers expenses towards deliveries and caesarean sections, but excludes expenses towards miscarriage (unless due to an accident) and lawful medical termination of pregnancy.
CONDITIONS: The policy covering the self and spouse must be in force when the benefit under this Section becomes payable, and both Self and Spouse should have been covered for a continuous period of 24 months under Star Comprehensive Insurance Policy.
POLICY REFERENCE: Section III, subsections a and b, and Specific Exclusions section.
----------------------------------------

TEST 2: What is the waiting period for pre-existing diseases?
🎯 Supervisor Decision: → coverage agent

📋 FINAL ANSWER:
----------------------------------------
COVERAGE STATU